# 01 — Phase 1: Position-Resolved Activation Patching

For every admitted item we patch each of the 144 heads from the **control** run into the
**conflict** run and record the normalised effect

$$e = \frac{d_{\text{patched}} - d_{\text{conflict}}}{d_{\text{control}} - d_{\text{conflict}}}$$

- `e = 0` → the head is irrelevant
- `e = 1` → this head alone carries the whole conflict effect
- `e < 0` → the head opposes

Normalising **per item** is required: averaging raw logit deltas across items re-introduces an
item-magnitude confound.

Readout is at `p_end`. The reported logit difference is a function of `resid_post[p_end]` and
nothing else, so a head's contribution to the *decision* is exactly its write there.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2

In [2]:
from circuit_conflict.utils import load_model
from circuit_conflict import dataset as D, pipeline as PL
import numpy as np

model = load_model()
df = D.load_prompts()
admitted = df[df.passes_precondition]
print(f"{len(admitted)//2} admitted items")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded gpt2 on mps
  n_layers=12, n_heads=12, d_model=768, d_head=64
73 admitted items


## Run the sweep

Items are batched by `(category, template, length, slot position)`. Grouping on template alone is
invalid when a template has a multi-token filler — those items differ in length and cannot share a
forward pass.

In [3]:
effects, behavioural = PL.run_patching(model, admitted)
for c, e in effects.items():
    np.save(PL.RESULTS / "phase1" / f"effects_{c}.npy", e)
    print(f"  {c}: effects {e.shape}")
behavioural.to_csv(PL.RESULTS / "phase1" / "behavioural.csv", index=False)

  A/A1: 25 items, seq=18


  B/B1: 11 items, seq=21


  B/B2: 12 items, seq=30


  C/C1: 18 items, seq=26


  C/C2: 1 items, seq=22


  C/C2: 3 items, seq=25


  C/C2: 1 items, seq=28


  C/C2: 2 items, seq=31


  A: effects (25, 12, 12)
  B: effects (23, 12, 12)
  C: effects (25, 12, 12)


## Behavioural summary

`d_conflict` > 0 means the slot/context channel wins the conflict; < 0 means the other channel does.

In [4]:
print(behavioural.groupby("category")[["d_conflict", "d_control", "swing"]].mean().round(2))

          d_conflict  d_control  swing
category                              
A               0.68      -2.59  -3.27
B              -0.37      -6.81  -6.44
C               3.90      -9.80 -13.70
